# Reddit Ticker Matching

將 01 階段清理後的 6 份 CSV 分塊處理，依 ticker 字典比對留言中的股票代碼，未匹配成功的留言刪除。

## 比對規則

- 使用成功的 00 階段 ticker 字典。
- 比對不區分大小寫。
- Token 定義為連續的 Unicode `\w+`：
  - `$AAPL`、`AAPL,`、`AAPL.` 可匹配。
  - `XAAPL`、`AAPL2`、`AAPL_test` 不匹配。
- 同一留言中的同一 ticker 只記錄一次，並按字母排序。
- 不對留言去重，即使 `id` 重複也保留。

## 資料處理

- 全程分塊處理。
- 跨 chunk 的唯一值統計使用 SQLite。
- 不使用逐列 × ticker dictionary 的 Python 迴圈。
- Unix timestamp 以 UTC 轉換日期相關欄位。
- 無法解析的日期保留並計入診斷，不直接刪除。

## Comment Enrichment

從原始 cleaned CSV 保留並整理：

- `id`
- `comment_id`
- `parent_id`
- `thread_id`
- UTC timestamp / date
- `year_month`
- `matched_tickers`

空白 thread 或無效日期維持 null，不額外過濾。

## 輸出

- 有匹配 ticker 的 comment-level Parquet
- ticker mention / pair Parquet
- `run_summary.json`

未匹配留言不寫入 Parquet，但納入 diagnostics。

每次執行建立新的 UTC 時間戳資料夾；成功後更新 `pipeline_runs.json`，失敗則不覆蓋既有 pointer。舊輸出保留。

In [3]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv, hashlib, importlib.metadata, json, platform, re, sqlite3, statistics, time, uuid
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import ahocorasick
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'Data/process').is_dir())

import sys
sys.path.insert(0,str(ROOT/'Notebook'))
from pipeline_io import new_run, publish, input_run, remove_work_database

INPUT_DIR = input_run(ROOT,'cleaned',ROOT/'Data/process')
UNIVERSE_DIR = input_run(ROOT,'universe',ROOT/'Data/process/universe_review')
DICTIONARY = UNIVERSE_DIR/('approx_common_stock_tickers.txt' if (UNIVERSE_DIR/'run_summary.json').exists() else 'approx_SHRCD_10_11_tickers.txt')
SOURCES = [INPUT_DIR / f'wallstreetbets_comment_part{i}.csv' for i in range(1, 7)]
assert all(p.is_file() for p in SOURCES), 'Expected exactly six named input files'
CHUNK_SIZE = 50_000
BENCHMARK_ROWS = 100_000
BENCHMARK_REPEATS = 3
COLUMNS = ['id', 'parent_id', 'author', 'created_utc', 'link_id', 'body']
RUN_DIR = new_run(ROOT,'matching')
INPUT_STATS = {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in SOURCES}
VERSIONS = {p: importlib.metadata.version(p) for p in ['pandas', 'pyarrow', 'pyahocorasick']}
print('Output:', RUN_DIR)
print('Versions:', VERSIONS)
display(pd.DataFrame([{'file': p.name, 'bytes': p.stat().st_size} for p in SOURCES]))

def read_chunks(path, nrows=None):
    return pd.read_csv(path, usecols=COLUMNS, dtype=str, keep_default_na=False,
                       na_filter=False, encoding='utf-8-sig', chunksize=CHUNK_SIZE,
                       nrows=nrows, on_bad_lines='error')

tickers = sorted({s.strip().upper() for s in DICTIONARY.read_text(encoding='utf-8-sig').splitlines() if s.strip()})
assert tickers and all(re.fullmatch(r'[A-Z]+', t) for t in tickers), 'Review tokenizer for a non-alphabetic dictionary'
TICKER_SET = frozenset(tickers)
print({'dictionary_rows': len(tickers), 'unique_tickers': len(TICKER_SET),
       'date_range': 'N/A: dictionary has no dates', 'unique_authors': 'N/A', 'unique_threads': 'N/A'})


Output: c:\Users\eagle\RedditStock\Data\process\02_ticker_matching\2026-09-22_03-52-09_UTC
Versions: {'pandas': '3.0.5', 'pyarrow': '22.0.0', 'pyahocorasick': '2.3.1'}


,file,bytes
0,wallstreetbets_comment_part1.csv,63729671
1,wallstreetbets_comment_part2.csv,97837151
2,wallstreetbets_comment_part3.csv,364124678
3,wallstreetbets_comment_part4.csv,544503864
4,wallstreetbets_comment_part5.csv,680250712
5,wallstreetbets_comment_part6.csv,358353292


{'dictionary_rows': 9804, 'unique_tickers': 9804, 'date_range': 'N/A: dictionary has no dates', 'unique_authors': 'N/A', 'unique_threads': 'N/A'}


## Matching Engine 與邊界檢查

提供兩種 matching engine：

- **Token + Hash Set**：先切分 token，再以 hash set 進行常數時間查找。
- **Aho–Corasick**：一次掃描全文搜尋所有 ticker，只有當匹配前後字元都不是 Unicode word character 時才接受。

兩種方法皆輸出排序後且不重複的 ticker 清單。


In [4]:
TOKEN_RE = re.compile(r'\w+', re.UNICODE)

def make_token_matcher(words):
    words = frozenset(words)
    def match(text):
        return sorted({token for token in TOKEN_RE.findall(text.upper()) if token in words})
    return match

def is_word(c):
    return c == '_' or c.isalnum()

def make_aho_matcher(words):
    automaton = ahocorasick.Automaton()
    for word in words:
        automaton.add_word(word, word)
    automaton.make_automaton()
    def match(text):
        text = text.upper()
        found = set()
        for end, word in automaton.iter(text):
            start = end - len(word) + 1
            if (start == 0 or not is_word(text[start-1])) and (end+1 == len(text) or not is_word(text[end+1])):
                found.add(word)
        return sorted(found)
    return match

build_times = {}
MATCHERS = {}
for name, factory in [('token_hash_set', make_token_matcher), ('aho_corasick', make_aho_matcher)]:
    started = time.perf_counter()
    MATCHERS[name] = factory(TICKER_SET)
    build_times[name] = time.perf_counter() - started

cases = {
    '$AAPL AAPL, AAPL.': ['AAPL'],
    'aapl / MsFt': ['AAPL', 'MSFT'],
    'XAAPL AAPLX AAPL2 2AAPL AAPL_test _AAPL': [],
    '中AAPL AAPLé': [],
    '($AAPL), MSFT. AAPL': ['AAPL', 'MSFT'],
    '': [], 'nothing here': [], 'AAPL\nMSFT': ['AAPL', 'MSFT'],
    'AA AAA AAAA': ['AAA'],
}
for factory in [make_token_matcher, make_aho_matcher]:
    test_match = factory({'AAPL', 'MSFT', 'AAA'})
    for body, expected in cases.items():
        assert test_match(body) == expected, (factory.__name__, body, test_match(body))
print('Boundary, case, repeated mention, multi-ticker and empty-body checks passed.')


Boundary, case, repeated mention, multi-ticker and empty-body checks passed.


## Benchmark

使用第一個輸入檔的前 100,000 筆資料測試 matching engine。

- 分塊讀取，僅保留單一 chunk 與兩組比對結果。
- 比較完整的逐列 matching 結果，確認兩者輸出一致。
- 選擇標準：3 次完整 100,000-row matching 時間。



In [5]:
def date_info(series):
    dt = pd.to_datetime(pd.to_numeric(series, errors='coerce'), unit='s', utc=True, errors='coerce')
    return {'date_min': None if dt.dropna().empty else dt.min().isoformat(),
            'date_max': None if dt.dropna().empty else dt.max().isoformat(),
            'invalid_date_count': int(dt.isna().sum())}

def local_diagnostics(frame):
    return {'row_count': len(frame), **date_info(frame['created_utc']),
            'unique_authors': int(frame.loc[frame.author.ne(''), 'author'].nunique()),
            'unique_threads': int(frame.loc[frame.link_id.ne(''), 'link_id'].nunique()),
            'missing_comment_id_count': int(frame.id.eq('').sum())}

timings = {name: [0.0] * BENCHMARK_REPEATS for name in MATCHERS}
benchmark_count = benchmark_matched = benchmark_multi = 0
benchmark_started = time.perf_counter()
for chunk_no, chunk in enumerate(read_chunks(SOURCES[0], nrows=BENCHMARK_ROWS), 1):
    bodies = chunk.body.tolist()
    for repeat in range(BENCHMARK_REPEATS):
        results = {}
        order = list(MATCHERS)
        if (repeat + chunk_no) % 2: order.reverse()
        for name in order:
            start = time.perf_counter()
            results[name] = [MATCHERS[name](body) for body in bodies]
            timings[name][repeat] += time.perf_counter() - start
        assert results['token_hash_set'] == results['aho_corasick'], 'Engines disagree; stop before formal run'
    matches = results['token_hash_set']
    mask = [bool(m) for m in matches]
    benchmark_count += len(chunk)
    benchmark_matched += sum(mask)
    benchmark_multi += sum(len(m) >= 2 for m in matches)
    print({'benchmark_chunk': chunk_no, 'input': local_diagnostics(chunk),
           'matched': local_diagnostics(chunk.loc[mask])}, flush=True)
    del bodies, results, matches, mask, chunk
assert benchmark_count == BENCHMARK_ROWS
benchmark_table = pd.DataFrame([{'method': name, 'rows': benchmark_count,
    'build_seconds': build_times[name], 'repeat_seconds': times,
    'median_seconds': statistics.median(times),
    'rows_per_second': benchmark_count / statistics.median(times)} for name, times in timings.items()])
SELECTED_METHOD = min(timings, key=lambda name: statistics.median(timings[name]))
match = MATCHERS[SELECTED_METHOD]
display(benchmark_table)
print({'selected_method': SELECTED_METHOD, 'matched_comments': benchmark_matched,
       'comments_with_ge2_tickers': benchmark_multi, 'exact_result_agreement': True})
# Benchmark is retained in notebook output and run_summary.json.


{'benchmark_chunk': 1, 'input': {'row_count': 50000, 'date_min': '1970-01-01T00:00:01+00:00', 'date_max': '2015-04-28T13:33:04+00:00', 'invalid_date_count': 13, 'unique_authors': 4300, 'unique_threads': 4247, 'missing_comment_id_count': 13}, 'matched': {'row_count': 39254, 'date_min': '2012-04-11T16:46:43+00:00', 'date_max': '2015-04-28T13:18:59+00:00', 'invalid_date_count': 9, 'unique_authors': 3857, 'unique_threads': 4108, 'missing_comment_id_count': 9}}
{'benchmark_chunk': 2, 'input': {'row_count': 50000, 'date_min': '1970-01-01T00:00:03+00:00', 'date_max': '2015-08-26T20:23:11+00:00', 'invalid_date_count': 1, 'unique_authors': 3727, 'unique_threads': 3675, 'missing_comment_id_count': 1}, 'matched': {'row_count': 35864, 'date_min': '2015-04-28T13:38:35+00:00', 'date_max': '2015-08-26T20:23:11+00:00', 'invalid_date_count': 1, 'unique_authors': 3210, 'unique_threads': 3510, 'missing_comment_id_count': 1}}


,method,rows,build_seconds,repeat_seconds,median_seconds,rows_per_second
0,token_hash_set,100000,0.000003,"[0.971318700001575, 0.8678123999852687, 0.8448...",0.867812,115232.278315
1,aho_corasick,100000,0.004082,"[1.122967200120911, 1.125861500040628, 1.21414...",1.125862,88820.871836


{'selected_method': 'token_hash_set', 'matched_comments': 75118, 'comments_with_ge2_tickers': 51480, 'exact_result_agreement': True}


## Parquet 輸出

每個輸入檔會產生：

- `*.comments.parquet`
- `*.mentions.parquet`

每處理完一個 chunk，就立即將匹配到的留言與 ticker mention 寫入 Parquet row group，並使用 Zstandard 壓縮。

- `matched_tickers` 儲存為字串列表。
- `created_utc` 保留原始字串格式。
- 寫入時先使用 `.partial` 暫存檔，成功關閉並驗證後才視為完成。
- 執行中斷則該次 run 不算完成，需在新目錄重新執行。
- 即使某輸入檔沒有任何匹配，也會建立有效的空 Parquet 檔。

## 統計

使用暫存 SQLite 計算每個輸入檔與各處理階段中的唯一值：

- 非空白 `author`
- 非空白 `thread`
- `ticker`

整體唯一值數量採跨檔案聯集計算，而不是直接加總各檔結果。

### 缺失 Comment ID

原始空白 comment ID 仍保留，並納入 diagnostics 統計。

兩種輸出都額外加入：

- `source_file`
- `source_row`：從 1 開始的原始 CSV 邏輯列號，不含 header

這兩個欄位只用於追溯來源，不會取代或虛構 Reddit ID。

In [6]:
def utc_datetimes(values):
    return pd.to_datetime(pd.to_numeric(values, errors='coerce'), unit='s', utc=True, errors='coerce').dt.as_unit('ns')

def thread_ids(values):
    normalized = values.fillna('').str.strip().str.lower().str.replace(r'^t3_', '', regex=True)
    return normalized.mask(normalized.eq(''), None)

test = utc_datetimes(pd.Series(['0', '1609459199', '1609459200', '0.5', 'bad', '']))
assert test.iloc[0].isoformat() == '1970-01-01T00:00:00+00:00'
assert test.iloc[1].strftime('%Y-%m') == '2020-12'
assert test.iloc[2].strftime('%Y-%m-%d') == '2021-01-01'
assert test.iloc[3].microsecond == 500000
assert test.iloc[4:].isna().all()
assert thread_ids(pd.Series(['t3_abc123', ' ABC123 ', '', None])).fillna('<null>').tolist() == ['abc123', 'abc123', '<null>', '<null>']
print('UTC and thread-normalization checks passed.')


COMMENT_SCHEMA = pa.schema([(c, pa.string()) for c in ['comment_id', 'author', 'created_utc', 'link_id', 'body']] +
                           [('matched_tickers', pa.list_(pa.string())), ('matched_ticker_count', pa.int32()), ('source_file', pa.string()), ('source_row', pa.int64())])
COMMENT_SCHEMA = pa.schema(list(COMMENT_SCHEMA) + [pa.field('id',pa.string()),pa.field('parent_id',pa.string()),pa.field('thread_id',pa.string()),pa.field('created_datetime_utc',pa.timestamp('ns',tz='UTC')),pa.field('created_date',pa.date32()),pa.field('year_month',pa.string())])
MENTION_SCHEMA = pa.schema([('comment_id', pa.string()), ('ticker', pa.string()), ('source_file', pa.string()), ('source_row', pa.int64())])
WORK_DB = RUN_DIR/'_work/diagnostics.sqlite'
WORK_DB.parent.mkdir()
db = sqlite3.connect(WORK_DB)
db.execute('PRAGMA cache_size=-32768')
db.execute('PRAGMA temp_store=FILE')
db.execute('CREATE TABLE uniques (file TEXT, stage TEXT, kind TEXT, value TEXT, PRIMARY KEY(file,stage,kind,value)) WITHOUT ROWID')

def record_uniques(file, stage, frame, ticker_values=()):
    for kind, values in [('authors', frame.author.unique()), ('threads', frame.link_id.unique()), ('tickers', ticker_values)]:
        db.executemany('INSERT OR IGNORE INTO uniques VALUES (?,?,?,?)',
                       ((file, stage, kind, str(value)) for value in values if value != ''))

def unique_count(stage, kind, file=None):
    if file is None:
        return db.execute('SELECT COUNT(DISTINCT value) FROM uniques WHERE stage=? AND kind=?', (stage, kind)).fetchone()[0]
    return db.execute('SELECT COUNT(*) FROM uniques WHERE file=? AND stage=? AND kind=?', (file, stage, kind)).fetchone()[0]

def merge_dates(stats, info, prefix):
    for suffix, func in [('date_min', min), ('date_max', max)]:
        key = prefix + suffix
        vals = [x for x in [stats.get(key), info[suffix]] if x is not None]
        stats[key] = func(vals) if vals else None
    key = prefix + 'invalid_date_count'
    stats[key] = stats.get(key, 0) + info['invalid_date_count']

def process_file(source):
    started = time.perf_counter()
    stats = {'file': source.name, 'raw_row_count': 0, 'matched_comment_count': 0,
             'comments_with_ge2_tickers': 0, 'mention_row_count': 0, 'raw_missing_comment_id_count': 0, 'matched_missing_comment_id_count': 0}
    comments_path = RUN_DIR / (source.stem + '.comments.parquet')
    mentions_path = RUN_DIR / (source.stem + '.mentions.parquet')
    cp = comments_path.with_suffix('.parquet.partial')
    mp = mentions_path.with_suffix('.parquet.partial')
    with pq.ParquetWriter(cp, COMMENT_SCHEMA, compression='zstd') as cw, pq.ParquetWriter(mp, MENTION_SCHEMA, compression='zstd') as mw:
        for chunk_no, chunk in enumerate(read_chunks(source), 1):
            chunk_started = time.perf_counter()
            hits = chunk.body.map(match)
            keep = hits.map(bool)
            matched = chunk.loc[keep].copy()
            matched['matched_tickers'] = hits.loc[keep]
            matched['matched_ticker_count'] = matched.matched_tickers.map(len)
            matched['source_file'] = source.name
            matched['source_row'] = matched.index.to_numpy() + 1
            matched['thread_id'] = thread_ids(matched.link_id)
            matched['created_datetime_utc'] = utc_datetimes(matched.created_utc)
            matched['created_date'] = matched.created_datetime_utc.dt.date
            matched['year_month'] = matched.created_datetime_utc.dt.strftime('%Y-%m')
            chunk_tickers = set(t for values in matched.matched_tickers for t in values)
            raw_diag = local_diagnostics(chunk)
            matched_diag = local_diagnostics(matched)
            enriched_diag = {'unique_threads':int(matched.thread_id.nunique()), 'missing_thread_count':int(matched.thread_id.isna().sum()), 'missing_parent_id_count':int(matched.parent_id.eq('').sum()), 'invalid_timestamp_count':int(matched.created_datetime_utc.isna().sum()), **date_info(matched.created_utc), 'row_count':len(matched), 'unique_authors':matched_diag['unique_authors']}
            stats['raw_missing_comment_id_count'] += raw_diag['missing_comment_id_count']
            stats['matched_missing_comment_id_count'] += matched_diag['missing_comment_id_count']
            stats['raw_row_count'] += len(chunk)
            stats['matched_comment_count'] += len(matched)
            stats['comments_with_ge2_tickers'] += int(matched.matched_ticker_count.ge(2).sum())
            merge_dates(stats, raw_diag, 'raw_')
            merge_dates(stats, matched_diag, 'matched_')
            record_uniques(source.name, 'raw', chunk)
            record_uniques(source.name, 'matched', matched, chunk_tickers)
            db.commit()
            if len(matched):
                output = matched.copy()
                output['comment_id'] = output['id']
                table = pa.Table.from_pandas(output, schema=COMMENT_SCHEMA, preserve_index=False)
                cw.write_table(table, row_group_size=CHUNK_SIZE)
                mentions = output[['comment_id', 'matched_tickers', 'source_file', 'source_row']].explode('matched_tickers').rename(columns={'matched_tickers': 'ticker'})
                assert len(mentions) == int(matched.matched_ticker_count.sum())
                mw.write_table(pa.Table.from_pandas(mentions, schema=MENTION_SCHEMA, preserve_index=False), row_group_size=CHUNK_SIZE)
                stats['mention_row_count'] += len(mentions)
                del output, table, mentions
            diagnostic = {'file': source.name, 'chunk': chunk_no, 'raw': raw_diag,
                          'matched': matched_diag, 'enriched':enriched_diag, 'unique_matched_tickers': len(chunk_tickers),
                          'seconds': round(time.perf_counter() - chunk_started, 3)}
            print(diagnostic, flush=True)
            del chunk, matched, hits, keep, chunk_tickers
    assert pq.read_metadata(cp).num_rows == stats['matched_comment_count']
    assert pq.read_metadata(mp).num_rows == stats['mention_row_count']
    cp.replace(comments_path)
    mp.replace(mentions_path)
    stats['match_rate'] = stats['matched_comment_count'] / stats['raw_row_count'] if stats['raw_row_count'] else 0.0
    for stage in ['raw', 'matched']:
        for kind in ['authors', 'threads']:
            stats[f'unique_{stage}_{kind}'] = unique_count(stage, kind, source.name)
    stats['unique_matched_tickers'] = unique_count('matched', 'tickers', source.name)
    stats['processing_seconds'] = time.perf_counter() - started
    display(pd.DataFrame([stats]))
    return stats


UTC and thread-normalization checks passed.


## Formal run: six files
The selected method processes all input rows, including the benchmark portion. Per-chunk diagnostics are printed and retained in notebook output. Per-file summaries are displayed immediately and consolidated in run_summary.json after success.


In [7]:
formal_started = time.perf_counter()
summaries = []
for source in SOURCES:
    print('Processing:', source.name, flush=True)
    summaries.append(process_file(source))
formal_seconds = time.perf_counter() - formal_started


Processing: wallstreetbets_comment_part1.csv
{'file': 'wallstreetbets_comment_part1.csv', 'chunk': 1, 'raw': {'row_count': 50000, 'date_min': '1970-01-01T00:00:01+00:00', 'date_max': '2015-04-28T13:33:04+00:00', 'invalid_date_count': 13, 'unique_authors': 4300, 'unique_threads': 4247, 'missing_comment_id_count': 13}, 'matched': {'row_count': 39254, 'date_min': '2012-04-11T16:46:43+00:00', 'date_max': '2015-04-28T13:18:59+00:00', 'invalid_date_count': 9, 'unique_authors': 3857, 'unique_threads': 4108, 'missing_comment_id_count': 9}, 'enriched': {'unique_threads': 4108, 'missing_thread_count': 9, 'missing_parent_id_count': 9, 'invalid_timestamp_count': 9, 'date_min': '2012-04-11T16:46:43+00:00', 'date_max': '2015-04-28T13:18:59+00:00', 'invalid_date_count': 9, 'row_count': 39254, 'unique_authors': 3857}, 'unique_matched_tickers': 1449, 'seconds': 0.852}
{'file': 'wallstreetbets_comment_part1.csv', 'chunk': 2, 'raw': {'row_count': 50000, 'date_min': '1970-01-01T00:00:03+00:00', 'date_max'

,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,raw_date_min,raw_date_max,raw_invalid_date_count,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part1.csv,310753,221516,145852,660525,23,14,1970-01-01T00:00:01+00:00,2016-04-25T12:28:09+00:00,23,2012-04-11T16:46:43+00:00,2016-04-25T12:28:09+00:00,14,0.712836,16596,20315,14439,19409,2615,5.871178


Processing: wallstreetbets_comment_part2.csv
{'file': 'wallstreetbets_comment_part2.csv', 'chunk': 1, 'raw': {'row_count': 50000, 'date_min': '2016-04-25T12:29:23+00:00', 'date_max': '2016-06-04T23:33:04+00:00', 'invalid_date_count': 0, 'unique_authors': 4924, 'unique_threads': 2994, 'missing_comment_id_count': 0}, 'matched': {'row_count': 33890, 'date_min': '2016-04-25T12:29:23+00:00', 'date_max': '2016-06-04T23:33:04+00:00', 'invalid_date_count': 0, 'unique_authors': 4128, 'unique_threads': 2804, 'missing_comment_id_count': 0}, 'enriched': {'unique_threads': 2804, 'missing_thread_count': 0, 'missing_parent_id_count': 0, 'invalid_timestamp_count': 0, 'date_min': '2016-04-25T12:29:23+00:00', 'date_max': '2016-06-04T23:33:04+00:00', 'invalid_date_count': 0, 'row_count': 33890, 'unique_authors': 4128}, 'unique_matched_tickers': 1296, 'seconds': 0.687}
{'file': 'wallstreetbets_comment_part2.csv', 'chunk': 2, 'raw': {'row_count': 50000, 'date_min': '2016-06-04T23:37:41+00:00', 'date_max': 

,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,raw_date_min,raw_date_max,raw_invalid_date_count,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part2.csv,575124,373356,229841,983294,0,0,2016-04-25T12:29:23+00:00,2017-04-01T06:13:54+00:00,0,2016-04-25T12:29:23+00:00,2017-04-01T06:13:54+00:00,0,0.649175,31225,33094,25508,31185,2883,9.919673


Processing: wallstreetbets_comment_part3.csv
{'file': 'wallstreetbets_comment_part3.csv', 'chunk': 1, 'raw': {'row_count': 50000, 'date_min': '2017-04-01T05:41:35+00:00', 'date_max': '2017-04-27T20:07:06+00:00', 'invalid_date_count': 0, 'unique_authors': 6592, 'unique_threads': 2561, 'missing_comment_id_count': 0}, 'matched': {'row_count': 31658, 'date_min': '2017-04-01T05:41:35+00:00', 'date_max': '2017-04-27T20:07:06+00:00', 'invalid_date_count': 0, 'unique_authors': 5293, 'unique_threads': 2337, 'missing_comment_id_count': 0}, 'enriched': {'unique_threads': 2337, 'missing_thread_count': 0, 'missing_parent_id_count': 0, 'invalid_timestamp_count': 0, 'date_min': '2017-04-01T05:41:35+00:00', 'date_max': '2017-04-27T20:07:06+00:00', 'invalid_date_count': 0, 'row_count': 31658, 'unique_authors': 5293}, 'unique_matched_tickers': 1135, 'seconds': 0.646}
{'file': 'wallstreetbets_comment_part3.csv', 'chunk': 2, 'raw': {'row_count': 50000, 'date_min': '2017-04-27T20:07:26+00:00', 'date_max': 

...

,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,raw_date_min,raw_date_max,raw_invalid_date_count,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part4.csv,3528708,2304903,1328747,5397420,0,0,2018-12-31T00:00:33+00:00,2020-02-01T00:00:15+00:00,0,2018-12-31T00:00:33+00:00,2020-02-01T00:00:15+00:00,0,0.653186,179220,48761,116851,45685,4112,59.876892


Processing: wallstreetbets_comment_part5.csv
{'file': 'wallstreetbets_comment_part5.csv', 'chunk': 1, 'raw': {'row_count': 50000, 'date_min': '2020-02-01T00:00:02+00:00', 'date_max': '2020-02-03T23:49:45+00:00', 'invalid_date_count': 0, 'unique_authors': 7556, 'unique_threads': 801, 'missing_comment_id_count': 0}, 'matched': {'row_count': 32604, 'date_min': '2020-02-01T00:00:02+00:00', 'date_max': '2020-02-03T23:49:43+00:00', 'invalid_date_count': 0, 'unique_authors': 6097, 'unique_threads': 723, 'missing_comment_id_count': 0}, 'enriched': {'unique_threads': 723, 'missing_thread_count': 0, 'missing_parent_id_count': 0, 'invalid_timestamp_count': 0, 'date_min': '2020-02-01T00:00:02+00:00', 'date_max': '2020-02-03T23:49:43+00:00', 'invalid_date_count': 0, 'row_count': 32604, 'unique_authors': 6097}, 'unique_matched_tickers': 923, 'seconds': 0.727}
{'file': 'wallstreetbets_comment_part5.csv', 'chunk': 2, 'raw': {'row_count': 50000, 'date_min': '2020-02-03T23:49:49+00:00', 'date_max': '202

,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,raw_date_min,raw_date_max,raw_invalid_date_count,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part5.csv,4458806,2808327,1588204,6396654,0,0,2020-02-01T00:00:02+00:00,2020-06-01T00:00:09+00:00,0,2020-02-01T00:00:02+00:00,2020-06-01T00:00:07+00:00,0,0.629838,159578,53068,128468,49828,4002,94.272795


Processing: wallstreetbets_comment_part6.csv
{'file': 'wallstreetbets_comment_part6.csv', 'chunk': 1, 'raw': {'row_count': 50000, 'date_min': '2020-06-01T00:00:01+00:00', 'date_max': '2020-06-02T05:36:27+00:00', 'invalid_date_count': 0, 'unique_authors': 9413, 'unique_threads': 1038, 'missing_comment_id_count': 0}, 'matched': {'row_count': 33101, 'date_min': '2020-06-01T00:00:04+00:00', 'date_max': '2020-06-02T05:36:27+00:00', 'invalid_date_count': 0, 'unique_authors': 7636, 'unique_threads': 967, 'missing_comment_id_count': 0}, 'enriched': {'unique_threads': 967, 'missing_thread_count': 0, 'missing_parent_id_count': 0, 'invalid_timestamp_count': 0, 'date_min': '2020-06-01T00:00:04+00:00', 'date_max': '2020-06-02T05:36:27+00:00', 'invalid_date_count': 0, 'row_count': 33101, 'unique_authors': 7636}, 'unique_matched_tickers': 1056, 'seconds': 0.913}
{'file': 'wallstreetbets_comment_part6.csv', 'chunk': 2, 'raw': {'row_count': 50000, 'date_min': '2020-06-02T05:36:28+00:00', 'date_max': '2

,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,raw_date_min,raw_date_max,raw_invalid_date_count,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part6.csv,2385062,1569148,886697,3580433,0,0,2020-06-01T00:00:01+00:00,2020-08-06T01:41:22+00:00,0,2020-06-01T00:00:04+00:00,2020-08-06T01:41:18+00:00,0,0.657907,97712,23872,79292,22840,3799,56.97965


## Combined statistics and output validation
Match rate is total matched input rows divided by total input rows. Unique authors, threads and tickers use exact unions across the six files. Parquet footer counts validate incremental writing without loading the output dataset into memory. A small preview reads only the first batch.


In [8]:
overall = {'file': 'ALL_FILES', 'processing_seconds': formal_seconds}
for key in ['raw_row_count', 'matched_comment_count', 'comments_with_ge2_tickers', 'mention_row_count',
            'raw_invalid_date_count', 'matched_invalid_date_count', 'raw_missing_comment_id_count', 'matched_missing_comment_id_count']:
    overall[key] = sum(s.get(key, 0) for s in summaries)
overall['match_rate'] = overall['matched_comment_count'] / overall['raw_row_count'] if overall['raw_row_count'] else 0.0
for stage in ['raw', 'matched']:
    for suffix, func in [('date_min', min), ('date_max', max)]:
        vals = [s[stage + '_' + suffix] for s in summaries if s.get(stage + '_' + suffix)]
        overall[stage + '_' + suffix] = func(vals) if vals else None
    for kind in ['authors', 'threads']:
        overall[f'unique_{stage}_{kind}'] = unique_count(stage, kind)
overall['unique_matched_tickers'] = unique_count('matched', 'tickers')
summary_table = pd.DataFrame(summaries + [overall])
display(summary_table)
assert INPUT_STATS == {str(p): (p.stat().st_size, p.stat().st_mtime_ns) for p in SOURCES}, 'Input file metadata changed'
assert sum(pq.read_metadata(RUN_DIR / (s.stem + '.comments.parquet')).num_rows for s in SOURCES) == overall['matched_comment_count']
assert sum(pq.read_metadata(RUN_DIR / (s.stem + '.mentions.parquet')).num_rows for s in SOURCES) == overall['mention_row_count']
manifest = {'status': 'complete', 'schema_version':2, 'comment_enrichment':True, 'selected_method': SELECTED_METHOD, 'chunk_size': CHUNK_SIZE,
            'benchmark_rows': BENCHMARK_ROWS, 'dictionary': str(DICTIONARY),
            'dictionary_sha256': hashlib.sha256(DICTIONARY.read_bytes()).hexdigest(),
            'input_metadata': INPUT_STATS, 'versions': VERSIONS, 'python': platform.python_version(),
            'case_policy': 'uppercase text and dictionary', 'token_policy': 'Unicode word tokens',
            'created_utc_output_type': 'original string', 'run_directory': str(RUN_DIR)}
db.close()
remove_work_database(WORK_DB)
for source in SOURCES:
    preview = next(pq.ParquetFile(RUN_DIR / (source.stem + '.comments.parquet')).iter_batches(batch_size=3), None)
    if preview is not None:
        display(preview.to_pandas())
        break
print('COMPLETE. Input sizes/mtimes unchanged; all Parquet row counts validated.')
print('Output:', RUN_DIR)

publish(ROOT,'matching',RUN_DIR,dict(manifest,files=summaries,overall=overall,benchmark=benchmark_table.to_dict(orient='records')))


,file,raw_row_count,matched_comment_count,comments_with_ge2_tickers,mention_row_count,raw_missing_comment_id_count,matched_missing_comment_id_count,raw_date_min,raw_date_max,raw_invalid_date_count,matched_date_min,matched_date_max,matched_invalid_date_count,match_rate,unique_raw_authors,unique_raw_threads,unique_matched_authors,unique_matched_threads,unique_matched_tickers,processing_seconds
0,wallstreetbets_comment_part1.csv,310753,221516,145852,660525,23,14,1970-01-01T00:00:01+00:00,2016-04-25T12:28:09+00:00,23,2012-04-11T16:46:43+00:00,2016-04-25T12:28:09+00:00,14,0.712836,16596,20315,14439,19409,2615,5.871178
1,wallstreetbets_comment_part2.csv,575124,373356,229841,983294,0,0,2016-04-25T12:29:23+00:00,2017-04-01T06:13:54+00:00,0,2016-04-25T12:29:23+00:00,2017-04-01T06:13:54+00:00,0,0.649175,31225,33094,25508,31185,2883,9.919673
2,wallstreetbets_comment_part3.csv,2218513,1437717,856162,3538335,0,0,2017-04-01T05:41:35+00:00,2018-12-31T00:05:11+00:00,0,2017-04-01T05:41:35+00:00,2018-12-31T00:05:03+00:00,0,0.648054,95173,98161,77224,92081,3720,37.491929
3,wallstreetbets_comment_part4.csv,3528708,2304903,1328747,5397420,0,0,2018-12-31T00:00:33+00:00,2020-02-01T00:00:15+00:00,0,2018-12-31T00:00:33+00:00,2020-02-01T00:00:15+00:00,0,0.653186,179220,48761,116851,45685,4112,59.876892
4,wallstreetbets_comment_part5.csv,4458806,2808327,1588204,6396654,0,0,2020-02-01T00:00:02+00:00,2020-06-01T00:00:09+00:00,0,2020-02-01T00:00:02+00:00,2020-06-01T00:00:07+00:00,0,0.629838,159578,53068,128468,49828,4002,94.272795
5,wallstreetbets_comment_part6.csv,2385062,1569148,886697,3580433,0,0,2020-06-01T00:00:01+00:00,2020-08-06T01:41:22+00:00,0,2020-06-01T00:00:04+00:00,2020-08-06T01:41:18+00:00,0,0.657907,97712,23872,79292,22840,3799,56.979650
6,ALL_FILES,13476966,8714967,5035503,20556661,23,14,1970-01-01T00:00:01+00:00,2020-08-06T01:41:22+00:00,23,2012-04-11T16:46:43+00:00,2020-08-06T01:41:18+00:00,14,0.646656,409612,273994,309073,258609,5682,264.461131


,comment_id,author,created_utc,link_id,body,matched_tickers,matched_ticker_count,source_file,source_row,id,parent_id,thread_id,created_datetime_utc,created_date,year_month
0,c4b0pvu,secondhandsondek,1334162803.0,t3_s4jw1,This is a fantastic idea Ill toss mine up in a...,"[BIT, BMO, JPM]",3,wallstreetbets_comment_part1.csv,1,c4b0pvu,t3_s4jw1,s4jw1,2012-04-11 16:46:43+00:00,2012-04-11,2012-04
1,c4b1fpf,glorysk87,1334165948.0,t3_s4jw1,INTC is on 417,[INTC],1,wallstreetbets_comment_part1.csv,2,c4b1fpf,t3_s4jw1,s4jw1,2012-04-11 17:39:08+00:00,2012-04-11,2012-04
2,c4b1rmm,entsportsjunkie,1334167351.0,t3_s4jw1,straddle call straddle put put put straddle ca...,"[CALL, FOR, MORE, ONE, TECH]",5,wallstreetbets_comment_part1.csv,3,c4b1rmm,t3_s4jw1,s4jw1,2012-04-11 18:02:31+00:00,2012-04-11,2012-04


COMPLETE. Input sizes/mtimes unchanged; all Parquet row counts validated.
Output: c:\Users\eagle\RedditStock\Data\process\02_ticker_matching\2026-09-22_03-52-09_UTC
